# NorthBridge Bank — Retail Banking Churn Analytics
## Simulated Consulting Engagement | Greater Toronto Area Market

**Role simulated:** Senior Data Analyst  
**Client:** NorthBridge Bank (fictional mid-sized Canadian retail bank)  
**Market:** Greater Toronto Area — 10,000 customers, FY2025  
**Tools:** Python · SQL · XGBoost · SHAP · Power BI · Logistic Regression

---

### What This Notebook Covers

This notebook walks through the core analytical work of a 7-phase retail banking analytics engagement:

| Phase | Focus | Key Output |
|---|---|---|
| 2 | Synthetic Dataset | 10,000-customer dataset calibrated to Toronto CMA signals |
| 3 | Customer Analytics | Churn by segment, CLV model, product penetration |
| 4 | Churn Model (XGBoost) | AUC 0.708, feature importance, risk scoring |
| 5 | Model Comparison & SHAP | Logistic regression baseline beats XGBoost on AUC; SHAP explains why |
| 6 | Model Risk Review | Fairness audit (5.7x disparate impact), calibration failure identified |
| 7 | Remediation | Calibration corrected, disparate impact reduced to 3.97x, CLV-weighted targeting |

> **Data note:** All customer data is fully synthetic. Demographics and churn rates were calibrated  
> to publicly available Toronto CMA signals (Statistics Canada 2021 Census, Canadian Payments data).  
> No real customer records were used.

---


## 0. Environment Setup

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.ticker as mtick
import warnings
warnings.filterwarnings("ignore")

# ML stack
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import LabelEncoder, StandardScaler
from sklearn.linear_model import LogisticRegression
from sklearn.calibration import CalibratedClassifierCV, calibration_curve
from sklearn.metrics import (
    roc_auc_score, average_precision_score, f1_score,
    confusion_matrix, classification_report, brier_score_loss,
    precision_score, recall_score, roc_curve
)
import xgboost as xgb
import shap

# Style
plt.rcParams.update({
    "figure.facecolor": "white",
    "axes.spines.top": False,
    "axes.spines.right": False,
    "axes.grid": True,
    "grid.alpha": 0.2,
    "font.family": "sans-serif",
})
NAVY, BLUE, TEAL, RED, AMBER, GREEN = "#1B3A6B", "#2563EB", "#0F766E", "#DC2626", "#D97706", "#16A34A"

np.random.seed(42)
print("Libraries loaded.")


---
## 1. Dataset Overview (Phase 2)

The synthetic dataset was built across 6 tables following a star schema, calibrated to Toronto CMA demographic data.  
Here we load the pre-built analytics master table — a customer-level feature table joining all 6 sources.


In [ ]:
# Load the pre-built analytics master table (joins all 6 raw tables)
df = pd.read_csv("data/analytics_master.csv")
print(f"Shape: {df.shape}")
print(f"Churn rate: {df.Churn_Flag.mean():.2%}")
df.head(3)


In [ ]:
# Portfolio snapshot
summary = df.groupby("Segment").agg(
    Customers=("Customer_ID", "count"),
    Churn_Rate=("Churn_Flag", "mean"),
    Avg_Products=("Product_Count", "mean"),
    Avg_Balance=("Total_Balance", "mean"),
    Avg_CLV=("CLV_5yr", "mean"),
).reset_index()
summary["Churn_Rate"] = summary["Churn_Rate"].map("{:.1%}".format)
summary["Avg_Products"] = summary["Avg_Products"].round(2)
summary["Avg_Balance"] = summary["Avg_Balance"].map("${:,.0f}".format)
summary["Avg_CLV"] = summary["Avg_CLV"].map("${:,.0f}".format)
summary.sort_values("Customers", ascending=False)


---
## 2. Customer Analytics (Phase 3)

### 2.1 Churn by Segment


In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(13, 5))

# Left: churn rate by segment
seg_churn = df.groupby("Segment")["Churn_Flag"].mean().sort_values(ascending=True)
colors = [RED if v > 0.15 else AMBER if v > 0.08 else GREEN for v in seg_churn.values]
axes[0].barh(seg_churn.index, seg_churn.values * 100, color=colors)
axes[0].xaxis.set_major_formatter(mtick.PercentFormatter())
axes[0].set_title("Churn Rate by Segment", fontweight="bold", color=NAVY)
for i, v in enumerate(seg_churn.values):
    axes[0].text(v * 100 + 0.3, i, f"{v:.1%}", va="center", fontsize=10)

# Right: 5-year CLV at risk from churned customers
clv_at_risk = (df[df.Churn_Flag == 1].groupby("Segment")["CLV_5yr"].sum() / 1e6).sort_values(ascending=True)
axes[1].barh(clv_at_risk.index, clv_at_risk.values, color=NAVY)
axes[1].set_xlabel("5-Year CLV at Risk ($M)")
axes[1].set_title("CLV at Risk from Churn by Segment", fontweight="bold", color=NAVY)
for i, v in enumerate(clv_at_risk.values):
    axes[1].text(v + 0.02, i, f"${v:.1f}M", va="center", fontsize=10)

plt.suptitle("Phase 3: Churn Concentration & Financial Exposure", fontsize=13, fontweight="bold", y=1.01)
plt.tight_layout()
plt.savefig("images/phase3_churn_clv.png", dpi=150, bbox_inches="tight")
plt.show()
print(f"Total CLV at risk: ${df[df.Churn_Flag==1]['CLV_5yr'].sum()/1e6:.1f}M")


### 2.2 Key Churn Drivers (Phase 3 Correlations)

In [ ]:
driver_cols = ["Product_Count", "Total_Balance", "Has_Direct_Deposit",
               "Monthly_Logins", "NPS_Score", "Years_With_Bank", "Complaint_Count"]
corrs = df[driver_cols + ["Churn_Flag"]].corr()["Churn_Flag"].drop("Churn_Flag").sort_values()

fig, ax = plt.subplots(figsize=(8, 5))
colors = [GREEN if v < 0 else RED for v in corrs.values]
ax.barh(corrs.index, corrs.values, color=colors)
ax.axvline(0, color="black", linewidth=0.8)
ax.set_title("Correlation with Churn Flag
(green = protective, red = risk-increasing)",
             fontweight="bold", color=NAVY)
ax.set_xlabel("Pearson Correlation Coefficient")
for i, v in enumerate(corrs.values):
    ax.text(v + (0.002 if v >= 0 else -0.002), i, f"{v:.3f}",
            va="center", ha="left" if v >= 0 else "right", fontsize=9)
plt.tight_layout()
plt.show()


---
## 3. Feature Engineering & Model Prep (Phase 4)


In [ ]:
EXCLUDE = ["Customer_ID", "Churn_Flag", "Churn_Risk_Score", "Churn_Risk_Tier",
           "CLV_5yr", "Primary_Bank_Score"]
CATEGORICAL = ["Segment", "Gender", "Occupation", "Income_Band",
               "Immigration_Status", "Region", "Relationship_Status"]

feature_cols = [c for c in df.columns if c not in EXCLUDE]
X = df[feature_cols].copy()
y = df["Churn_Flag"].copy()

for col in CATEGORICAL:
    le = LabelEncoder()
    X[col] = le.fit_transform(X[col].astype(str))

for c in [col for col in X.columns if X[col].dtype == bool]:
    X[c] = X[c].astype(int)

X_train, X_test, y_train, y_test, idx_train, idx_test = train_test_split(
    X, y, df.index, test_size=0.25, random_state=42, stratify=y
)
test_meta = df.loc[idx_test, ["Customer_ID", "Segment", "Immigration_Status", "CLV_5yr"]].reset_index(drop=True)
y_test_arr = y_test.reset_index(drop=True).values

print(f"Features: {X.shape[1]}  |  Train: {len(X_train)}  |  Test: {len(X_test)}")
print(f"Train churn rate: {y_train.mean():.2%}  |  Test churn rate: {y_test.mean():.2%}")


---
## 4. Churn Prediction Models (Phases 4–5)

### 4.1 Logistic Regression (Baseline)


In [ ]:
scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.transform(X_test)

logit = LogisticRegression(max_iter=1000, class_weight="balanced", random_state=42, C=1.0)
logit.fit(X_train_scaled, y_train)
logit_proba = logit.predict_proba(X_test_scaled)[:, 1]

logit_auc = roc_auc_score(y_test, logit_proba)
logit_ap = average_precision_score(y_test, logit_proba)
print(f"Logistic Regression — AUC: {logit_auc:.4f} | Avg Precision: {logit_ap:.4f}")


### 4.2 XGBoost Classifier

In [ ]:
scale_pos_weight = (y_train == 0).sum() / (y_train == 1).sum()
xgb_model = xgb.XGBClassifier(
    n_estimators=300, max_depth=4, learning_rate=0.05,
    subsample=0.85, colsample_bytree=0.85, min_child_weight=3,
    gamma=0.1, reg_alpha=0.1, reg_lambda=1.0,
    scale_pos_weight=scale_pos_weight, eval_metric="auc",
    random_state=42, n_jobs=-1,
)
xgb_model.fit(X_train, y_train)
xgb_proba = xgb_model.predict_proba(X_test)[:, 1]

xgb_auc = roc_auc_score(y_test, xgb_proba)
xgb_ap = average_precision_score(y_test, xgb_proba)
print(f"XGBoost          — AUC: {xgb_auc:.4f} | Avg Precision: {xgb_ap:.4f}")


### 4.3 Model Comparison: ROC Curves

In [ ]:
fpr_l, tpr_l, _ = roc_curve(y_test, logit_proba)
fpr_x, tpr_x, _ = roc_curve(y_test, xgb_proba)

plt.figure(figsize=(7, 6))
plt.plot(fpr_l, tpr_l, color="#94A3B8", linewidth=2.2,
         label=f"Logistic Regression (AUC = {logit_auc:.3f})")
plt.plot(fpr_x, tpr_x, color=BLUE, linewidth=2.5,
         label=f"XGBoost (AUC = {xgb_auc:.3f})")
plt.plot([0, 1], [0, 1], color="#CBD5E1", linestyle="--", linewidth=1)
plt.xlabel("False Positive Rate")
plt.ylabel("True Positive Rate")
plt.title("Model Comparison: ROC Curves
(Phase 5 finding: logistic regression edges out XGBoost)",
          fontweight="bold", color=NAVY)
plt.legend(loc="lower right")
plt.tight_layout()
plt.savefig("images/roc_comparison.png", dpi=150, bbox_inches="tight")
plt.show()

print("\nKey finding: the simpler logistic regression model outperforms XGBoost on AUC,")
print("average precision, and F1. This reflects the dataset's largely linear churn signal.")
print("XGBoost wins on recall — catching more true churners — which matters when the cost")
print("of a missed churner outweighs the cost of a false contact.")


---
## 5. SHAP Explainability (Phase 5)

SHAP (SHapley Additive exPlanations) answers not just *what* drives churn globally,  
but *why* the model made a specific prediction for each individual customer.


In [ ]:
explainer = shap.TreeExplainer(xgb_model)
sample_idx = np.random.choice(len(X_test), size=800, replace=False)
X_shap = X_test.iloc[sample_idx].reset_index(drop=True)
shap_values = explainer.shap_values(X_shap)

# Global summary
plt.figure(figsize=(8, 7))
shap.summary_plot(shap_values, X_shap, show=False, plot_size=None)
plt.title("SHAP Summary — Global Feature Impact on Churn Prediction",
          fontweight="bold", color=NAVY, pad=15)
plt.tight_layout()
plt.savefig("images/shap_summary.png", dpi=150, bbox_inches="tight")
plt.show()


In [ ]:
# Reading the SHAP summary plot:
# - Each row = one feature; each dot = one customer
# - Position (x-axis) = SHAP value (how much that feature pushed the prediction)
# - Color (red = high feature value, blue = low feature value)

# Key insight from the beeswarm:
# Low monthly deposits (blue dots) → positive SHAP → higher predicted churn risk
# This is the strongest and most consistent signal in the model

# Individual explanation: high-risk customer
xgb_proba_sample = xgb_model.predict_proba(X_shap)[:, 1]
high_risk_idx = int(np.argmax(xgb_proba_sample))

exp = shap.Explanation(
    values=shap_values[high_risk_idx],
    base_values=explainer.expected_value,
    data=X_shap.iloc[high_risk_idx],
    feature_names=X.columns.tolist()
)
plt.figure(figsize=(8, 5))
shap.plots.waterfall(exp, show=False, max_display=10)
plt.title(f"SHAP Waterfall — High-Risk Customer (predicted churn: {xgb_proba_sample[high_risk_idx]:.1%})",
          fontweight="bold", color=NAVY)
plt.tight_layout()
plt.savefig("images/shap_waterfall_high_risk.png", dpi=150, bbox_inches="tight")
plt.show()


---
## 6. Fairness Audit (Phase 6)

Aggregate metrics hide how a model treats different groups.  
At a single global threshold, the model's flagging rate varied by **5.7x across immigration status groups**  
— far beyond what the actual 3.4x difference in churn rates would justify.


In [ ]:
# Establish an equivalent global threshold on this run's uncalibrated probabilities
thresholds = np.arange(0.01, 0.99, 0.005)
global_flagged_target = (logit_proba >= 0.58).mean()  # Phase 5 F1-optimal rate
best_t = min(thresholds, key=lambda t: abs((logit_proba >= t).mean() - global_flagged_target))
GLOBAL_THRESHOLD = round(float(best_t), 3)
global_pred = (logit_proba >= GLOBAL_THRESHOLD).astype(int)

# Compute flagging rate per immigration status group
imm_groups = test_meta["Immigration_Status"].unique()
rows = []
for g in imm_groups:
    mask = (test_meta["Immigration_Status"] == g).values
    rows.append({
        "Group": g,
        "N": int(mask.sum()),
        "Actual Churn Rate": f"{y_test_arr[mask].mean():.1%}",
        "Flagged Rate (global threshold)": f"{global_pred[mask].mean():.1%}",
    })
fairness_before = pd.DataFrame(rows).sort_values("Flagged Rate (global threshold)", ascending=False)

flagged_rates = {r["Group"]: float(r["Flagged Rate (global threshold)"].strip("%")) / 100
                 for _, r in fairness_before.iterrows()}
ratio = max(flagged_rates.values()) / min(flagged_rates.values())
print(f"Disparate impact ratio: {ratio:.2f}x")
print()
print(fairness_before.to_string(index=False))


---
## 7. Model Remediation (Phase 7)

Phase 6 found three defects. This section implements a fix for each and re-measures the result.

### 7.1 Calibration Correction (Isotonic Regression)


In [ ]:
# Split training data: 70% for model fitting, 30% for calibration
X_fit, X_cal, y_fit, y_cal = train_test_split(
    X_train_scaled, y_train, test_size=0.30, random_state=42, stratify=y_train
)
base_logit = LogisticRegression(max_iter=1000, class_weight="balanced", random_state=42, C=1.0)
base_logit.fit(X_fit, y_fit)
uncal_proba = base_logit.predict_proba(X_test_scaled)[:, 1]

try:
    from sklearn.frozen import FrozenEstimator
    cal_model = CalibratedClassifierCV(FrozenEstimator(base_logit), method="isotonic")
except ImportError:
    cal_model = CalibratedClassifierCV(base_logit, method="isotonic", cv="prefit")
cal_model.fit(X_cal, y_cal)
cal_proba = cal_model.predict_proba(X_test_scaled)[:, 1]

brier_before = brier_score_loss(y_test_arr, uncal_proba)
brier_after = brier_score_loss(y_test_arr, cal_proba)

fig, ax = plt.subplots(figsize=(7, 6))
uncal_frac, uncal_mean = calibration_curve(y_test_arr, uncal_proba, n_bins=10, strategy="quantile")
cal_frac, cal_mean = calibration_curve(y_test_arr, cal_proba, n_bins=10, strategy="quantile")
ax.plot([0, 1], [0, 1], color="#CBD5E1", linestyle="--", linewidth=1, label="Perfect calibration")
ax.plot(uncal_mean, uncal_frac, "o-", color=RED, linewidth=2, label=f"Before (Brier={brier_before:.3f})")
ax.plot(cal_mean, cal_frac, "o-", color=GREEN, linewidth=2, label=f"After (Brier={brier_after:.3f})")
ax.set_xlabel("Mean Predicted Probability")
ax.set_ylabel("Observed Churn Rate")
ax.set_title(f"Calibration Correction\nBrier score improved {(1-brier_after/brier_before)*100:.1f}%",
             fontweight="bold", color=NAVY)
ax.legend()
plt.tight_layout()
plt.show()
print(f"Brier score: {brier_before:.4f} → {brier_after:.4f}  ({(1-brier_after/brier_before)*100:.1f}% improvement)")


### 7.2 Segment-Aware Thresholds (Fairness Remediation)

In [ ]:
def segment_threshold(proba, y_true, mask, multiplier=1.3):
    """Find threshold so flagged rate ≈ actual_churn_rate * multiplier."""
    actual = y_true[mask].mean()
    target = min(actual * multiplier, 0.95)
    best_t = min(np.arange(0.02, 0.97, 0.01),
                 key=lambda t: abs((proba[mask] >= t).mean() - target))
    return round(float(best_t), 2)

# Compute equivalent global threshold on calibrated probabilities
cal_global_t = min(np.arange(0.01, 0.99, 0.005),
                   key=lambda t: abs((cal_proba >= t).mean() - global_flagged_target))
cal_global_pred = (cal_proba >= cal_global_t).astype(int)

# Segment-aware thresholds by immigration status
imm_thresholds = {
    g: segment_threshold(cal_proba, y_test_arr, (test_meta["Immigration_Status"] == g).values)
    for g in imm_groups
}
fixed_pred = np.zeros(len(cal_proba), dtype=int)
for g, t in imm_thresholds.items():
    mask = (test_meta["Immigration_Status"] == g).values
    fixed_pred[mask] = (cal_proba[mask] >= t).astype(int)

# Compare before / after
results = []
for g in sorted(imm_groups):
    mask = (test_meta["Immigration_Status"] == g).values
    results.append({
        "Group": g,
        "Actual Churn": f"{y_test_arr[mask].mean():.1%}",
        "Flagged Before": f"{cal_global_pred[mask].mean():.1%}",
        "Flagged After": f"{fixed_pred[mask].mean():.1%}",
    })

result_df = pd.DataFrame(results)
print(result_df.to_string(index=False))

before_rates = {g: cal_global_pred[(test_meta["Immigration_Status"]==g).values].mean() for g in imm_groups}
after_rates  = {g: fixed_pred[(test_meta["Immigration_Status"]==g).values].mean() for g in imm_groups}
ratio_before = max(before_rates.values()) / min(before_rates.values())
ratio_after  = max(after_rates.values()) / min(after_rates.values())
print(f"\nDisparate impact ratio: {ratio_before:.2f}x → {ratio_after:.2f}x")
print("(30% reduction; gap narrowed but not fully closed — see Phase 6/7 reports for full analysis)")


### 7.3 Capacity-Constrained, CLV-Weighted Targeting

In [ ]:
CAPACITY = 75  # proportional monthly capacity for this 2,500-customer test set

priority = test_meta.copy()
priority["Cal_Churn_Prob"] = cal_proba
priority["Actual_Churn"] = y_test_arr
priority["Priority_Score"] = priority["Cal_Churn_Prob"] * priority["CLV_5yr"]

# CLV-weighted ranking
clv_ranked = priority.sort_values("Priority_Score", ascending=False).head(CAPACITY)
# Flat probability ranking (same capacity)
flat_ranked = priority.sort_values("Cal_Churn_Prob", ascending=False).head(CAPACITY)

clv_caught  = int(clv_ranked["Actual_Churn"].sum())
flat_caught = int(flat_ranked["Actual_Churn"].sum())
clv_value   = clv_ranked.loc[clv_ranked["Actual_Churn"]==1, "CLV_5yr"].sum()
flat_value  = flat_ranked.loc[flat_ranked["Actual_Churn"]==1, "CLV_5yr"].sum()

print(f"At {CAPACITY} monthly contacts (test-set scale):")
print(f"  CLV-weighted: {clv_caught} actual churners | ${clv_value:,.0f} CLV of churners")
print(f"  Flat prob:    {flat_caught} actual churners | ${flat_value:,.0f} CLV of churners")
print(f"  CLV uplift: +{(clv_value/flat_value - 1)*100:.1f}% more value captured per contact")
print()
print("Note: CLV-weighting catches fewer individual churners but targets higher-value ones.")
print("Whether that tradeoff is correct depends on the retention team's business objective.")

# Capacity curve
caps = np.arange(10, 600, 10)
clv_curve  = [priority.sort_values("Priority_Score", ascending=False).head(c)
              .loc[lambda d: d.Actual_Churn==1, "CLV_5yr"].sum() for c in caps]
flat_curve = [priority.sort_values("Cal_Churn_Prob", ascending=False).head(c)
              .loc[lambda d: d.Actual_Churn==1, "CLV_5yr"].sum() for c in caps]

plt.figure(figsize=(7.5, 5.5))
plt.plot(caps, np.array(clv_curve)/1e3, color=TEAL, linewidth=2.3, label="CLV-weighted ranking")
plt.plot(caps, np.array(flat_curve)/1e3, color="#94A3B8", linewidth=2.3, label="Flat probability ranking")
plt.axvline(CAPACITY, color=RED, linestyle="--", linewidth=1.3, label=f"Realistic capacity ({CAPACITY})")
plt.xlabel("Monthly Contact Capacity (test-set scale)")
plt.ylabel("CLV of Actual Churners Captured ($K)")
plt.title("Capacity-Constrained Targeting: CLV-Weighted vs. Flat Ranking",
          fontweight="bold", color=NAVY)
plt.legend(loc="lower right")
plt.tight_layout()
plt.savefig("images/capacity_targeting.png", dpi=150, bbox_inches="tight")
plt.show()


---
## 8. Summary of Results

| Phase | Finding | Status |
|---|---|---|
| 3 | Students (26.8%) and Newcomers (21.8%) drive 68% of all churn | ✅ Quantified |
| 3 | $14.6M in 5-year CLV at risk across churned customers | ✅ Quantified |
| 4 | XGBoost churn model: AUC 0.708 | ✅ Trained |
| 5 | Logistic regression baseline outperforms XGBoost on AUC (0.730 vs 0.708) | ✅ Honest finding |
| 5 | SHAP: monthly deposit behavior is the single strongest churn driver | ✅ Explained |
| 6 | Model flags 95.7% of International Students vs 16.2% of Canadian Born — 5.7x disparate impact | ⚠️ Identified |
| 6 | Both models systematically overconfident; Brier 0.21 | ⚠️ Identified |
| 7 | Isotonic calibration reduces Brier by 54% | ✅ Resolved |
| 7 | Segment-aware thresholds reduce disparate impact to 3.97x | ⚠️ Partial |
| 7 | CLV-weighted targeting captures 165.8% more value per contact vs. flat ranking | ✅ Resolved |

### Deployment Recommendation

**Conditional deployment**: Launch with calibrated model + CLV-weighted targeting for  
Family, Affluent, and Young Professional segments, where the fairness gap is smallest.  
Hold Student and Newcomer segments in manual review until the remaining disparate  
impact gap is closed via fairness-constrained retraining.

---

### Technical Stack Used in This Project

| Tool | Use |
|---|---|
| Python (pandas, numpy) | Data generation, feature engineering, analytics |
| scikit-learn | Logistic regression, calibration, train-test split |
| XGBoost | Churn prediction model |
| SHAP | Global and per-customer model explanations |
| SQL (ANSI) | Production analytics layer (see `/sql/`) |
| Power BI (DAX) | Executive dashboard data model (see `/powerbi/`) |
| Matplotlib | All visualizations |

---
*Matias — NorthBridge Bank Retail Analytics Portfolio Project*
